Neste notebook, irei responder às perguntas de negócio mapeadas no projeto. 
Para demonstrar versatilidade, apresentarei a lógica original pensada em SQL (detalhada no arquivo insights-negocios.md) seguida da sua respectiva execução em Python utilizando a biblioteca Pandas.

In [2]:
# A Célula de Importação
import pandas as pd
import sys

# Adiciona o diretório raiz ao caminho para o Python encontrar o cenxao.py
sys.path.append('..')
from conexao import obter_dados



In [3]:
# Carregando as tabelas principais
df_songs = obter_dados("SELECT * FROM dbo.Songs;")
df_artists = obter_dados("SELECT * FROM dbo.Artists;")
df_users = obter_dados("SELECT * FROM dbo.Users;")
df_songplays = obter_dados("SELECT * FROM dbo.SongPlays;")
df_genre = obter_dados("SELECT * FROM dbo.Genre;")
df_location = obter_dados("SELECT * FROM dbo.Location;")
df_albums = obter_dados("SELECT * FROM dbo.Albums;")
df_labels = obter_dados("SELECT * FROM dbo.Labels;")

## 1. Quais são as 10 músicas mais reproduzidas?

**Pergunta de negócio:** Quais conteúdos geram mais engajamento na plataforma?

```sql
SELECT TOP 10 s.Title AS Musica,
    ar.Name AS Artista,
    COUNT(sp.SongPlayID) AS TotalReproducoes
FROM dbo.SongPlays sp
    JOIN dbo.Songs s ON s.SongID = sp.SongID
    JOIN dbo.Artists ar ON ar.ArtistID = s.ArtistID
GROUP BY s.Title, ar.Name
ORDER BY TotalReproducoes DESC;
```

**Por que importa:** base para sistemas de recomendação e decisões de curadoria editorial.

In [ ]:
top_10_musicas = (
    pd.merge(df_songplays, df_songs, on='SongID', how='inner')
    .merge(df_artists, on='ArtistID', how='inner')
    .groupby(['Title', 'Name'])['SongPlayID']
    .count()
    .reset_index()
    .rename(columns={'Title': 'Musica', 'Name': 'Artista', 'SongPlayID': 'TotalReproducoes'})
    .sort_values(by='TotalReproducoes', ascending=False)
    .head(10)
)

# Exibindo o resultado final
top_10_musicas

## 2. Quais artistas acumulam mais reproduções no total?

**Pergunta de negócio:** Quem são os artistas âncora da plataforma?

```sql
SELECT TOP 10 ar.Name AS Artista,
    COUNT(sp.SongPlayID) AS TotalReproducoes,
    COUNT(DISTINCT s.SongID) AS QtdMusicas
FROM dbo.SongPlays sp
    JOIN dbo.Songs s ON s.SongID = sp.SongID
    JOIN dbo.Artists ar ON ar.ArtistID = s.ArtistID
GROUP BY ar.Name
ORDER BY TotalReproducoes DESC;
```

**Por que importa:** orienta negociações de licenciamento e destaque na plataforma.

In [ ]:
top_10_artistas = (
    pd.merge(df_songplays, df_songs, on='SongID', how='inner')
    .merge(df_artists, on='ArtistID', how='inner')
    .groupby('Name')
    .agg(
        TotalReproducoes=('SongPlayID', 'count'),
        QtdMusicas=('SongID', 'nunique')
    )
    .reset_index()
    .rename(columns={'Name' : 'Artistas'})
    .sort_values(by='TotalReproducoes', ascending=False)
    .head(10)
)

# Exibindo o resultado no notebook
top_10_artistas

## 3. Quais gêneros musicais são mais consumidos?

**Pergunta de negócio:** Quais gêneros devem receber mais investimento em catálogo?

```sql
SELECT g.Name AS Genero,
    COUNT(sp.SongPlayID) AS TotalReproducoes,
    COUNT(DISTINCT s.SongID) AS QtdMusicas
FROM dbo.SongPlays sp
    JOIN dbo.Songs s ON s.SongID = sp.SongID
    JOIN dbo.Genre g ON g.GenreID = s.GenreID
GROUP BY g.Name
ORDER BY TotalReproducoes DESC;
```

**Por que importa:** mostra onde concentrar aquisição de novos conteúdos.

In [3]:
top_generos = (
    pd.merge(df_songplays, df_songs, on='SongID', how='inner')
    .merge(df_genre, on='GenreID', how='inner')
    .groupby('Name')
    .agg(
        TotalReproducoes=('SongPlayID', 'count'),
        QtdMusicas=('SongID', 'nunique')
    )
    .reset_index()
    .rename(columns={'Name' : 'Genero'})
    .sort_values(by='TotalReproducoes', ascending=False)
)

top_generos

,Genero,TotalReproducoes,QtdMusicas
3,Pop,3573,55
1,Hip-Hop,2019,32
4,R&B,1100,17
5,Rock,517,8
0,Electronic,510,8
2,Indie,281,4


## 4. Qual o volume de reproduções por mês?

**Pergunta de negócio:** O engajamento está crescendo, estável ou caindo ao longo do tempo?

```sql
SELECT YEAR(sp.StartTime) AS Ano,
    MONTH(sp.StartTime) AS Mes,
    COUNT(sp.SongPlayID) AS TotalReproducoes
FROM dbo.SongPlays sp
GROUP BY YEAR(sp.StartTime),
    MONTH(sp.StartTime)
ORDER BY Ano, Mes;
```

**Por que importa:** série temporal essencial para análise de tendência e sazonalidade.

In [ ]:
reproducoes_por_mes = (
    df_songplays
    # Cria novas colunas 'Ano' e 'Mes' extraindo a informação da data
    .assign(
        Ano=lambda df:pd.to_datetime(df['StartTime']).dt.year,
        Mes=lambda df:pd.to_datetime(df['StartTime']).dt.month
    )
    .groupby (['Ano', 'Mes'])
    .agg(TotalReprucoes=('SongPlayID', 'count'))
    .reset_index()
    .sort_values(by=['Ano', 'Mes'])
)

reproducoes_por_mes

,Ano,Mes,TotalReprucoes
0,2022,1,210
1,2022,2,207
2,2022,3,196
3,2022,4,213
4,2022,5,219
5,2022,6,241
6,2022,7,223
7,2022,8,244
8,2022,9,216
9,2022,10,215


## 5. Quais países têm mais reproduções?

**Pergunta de negócio:** Onde está concentrada a audiência da plataforma?

```sql
SELECT l.Country AS Pais,
    COUNT(sp.SongPlayID) AS TotalReproducoes,
    COUNT(DISTINCT sp.UserID) AS UsuariosAtivos
FROM dbo.SongPlays sp
    JOIN dbo.Location l ON l.LocationID = sp.LocationID
GROUP BY l.Country
ORDER BY TotalReproducoes DESC;
```

**Por que importa:** direciona estratégias de expansão e localização de conteúdo.

In [9]:
reproducoes_por_pais=(
    pd.merge(df_songplays, df_location, on='LocationID', how='inner')
    .groupby('Country')
    .agg(
        TotalReproducoes=('SongPlayID', 'count'),
        UsuariosAtivos=('UserID', 'nunique')
    )
    .reset_index()
    .sort_values(by='TotalReproducoes', ascending=False)
)

reproducoes_por_pais

,Country,TotalReproducoes,UsuariosAtivos
10,Mexico,443,316
2,Brazil,428,317
15,South Africa,427,296
8,India,421,308
9,Japan,419,309
1,Australia,415,309
7,Germany,412,297
4,Chile,409,301
6,France,405,289
16,South Korea,403,297


## 6. Quais álbuns têm a maior média de reproduções por faixa?

**Pergunta de negócio:** Quais álbuns performam bem de forma consistente - não só pelo hit isolado?

```sql
SELECT TOP 10 al.Name AS Album,
    ar.Name AS Artista,
    COUNT(DISTINCT s.SongID) AS QtdFaixas,
    COUNT(sp.SongPlayID) AS TotalReproducoes,
    COUNT(sp.SongPlayID) / COUNT(DISTINCT s.SongID) AS MediaPorFaixa
FROM dbo.Albums al
    JOIN dbo.Artists ar ON ar.ArtistID = al.ArtistID
    JOIN dbo.Songs s ON s.AlbumID = al.AlbumID
    JOIN dbo.SongPlays sp ON sp.SongID = s.SongID
GROUP BY al.Name, ar.Name
HAVING COUNT(DISTINCT s.SongID) >= 3
ORDER BY MediaPorFaixa DESC;
```

**Por que importa:** identifica álbuns com qualidade uniforme, não só com um single popular.

In [17]:
# Garantindo os nomes limpos
df_albums_limpo = df_albums.rename(columns={'Name': 'Album'})
df_artists_limpo = df_artists.rename(columns={'Name': 'Artista'})

top_10_albuns_consistentes = (
    pd.merge(df_songplays, df_songs, on='SongID', how='inner')
    .merge(df_albums_limpo, on=['AlbumID', 'ArtistID'], how='inner')
    .merge(df_artists_limpo, on='ArtistID', how='inner')
    .groupby(['Album', 'Artista'])
    .agg(
        QtdFaixas=('SongID', 'nunique'),
        TotalReproducoes=('SongPlayID', 'count')
    )
    .reset_index()
    .query('QtdFaixas >= 3')
    .assign(MediaPorFaixa=lambda df: df['TotalReproducoes'] / df['QtdFaixas'])
    .sort_values(by='MediaPorFaixa', ascending=False)
    .head(10)
)

# Exibindo o resultado no notebook
top_10_albuns_consistentes

,Album,Artista,QtdFaixas,TotalReproducoes,MediaPorFaixa
27,When We All Fall Asleep,Billie Eilish,4,285,71.25
10,Future Nostalgia,Dua Lipa,5,352,70.40
21,Norman Fucking Rockwell!,Lana Del Rey,4,281,70.25
12,Harry's House,Harry Styles,4,281,70.25
22,Planet Her,Doja Cat,4,279,69.75
6,Certified Lover Boy,Drake,4,270,67.50
7,Chromatica,Lady Gaga,4,269,67.25
4,After Hours,The Weeknd,5,336,67.20
15,Justice,Justin Bieber,4,268,67.00
23,SOS,SZA,4,268,67.00


## 7. Quais gravadoras têm mais músicas no catálogo?

**Pergunta de negócio:** Existe concentração de catálogo em poucas gravadoras - um risco para a plataforma?

```sql
SELECT lb.Name AS Gravadora,
    COUNT(DISTINCT s.SongID) AS QtdMusicas,
    COUNT(DISTINCT al.AlbumID) AS QtdAlbuns,
    CAST(
        COUNT(DISTINCT s.SongID) * 100.0 / SUM(COUNT(DISTINCT s.SongID)) OVER () AS DECIMAL(5, 2)
    ) AS PercentualCatalogo
FROM dbo.Labels lb
    JOIN dbo.Albums al ON al.LabelID = lb.LabelID
    JOIN dbo.Songs s ON s.AlbumID = al.AlbumID
GROUP BY lb.Name
ORDER BY QtdMusicas DESC;
```

**Por que importa:** concentração em poucas gravadoras representa risco contratual e de negócio.

In [6]:
# Renomeando previamente para evitar conflitos e já deixar no formato final
df_labels_limpo = df_labels.rename(columns={'Name': 'Gravadora'})

gravadoras_catalogo = (
    pd.merge(df_songs, df_albums, on='AlbumID', how='inner')
    .merge(df_labels_limpo, on='LabelID', how='inner')
    .groupby('Gravadora')
    .agg(
        QtdMusicas=('SongID', 'nunique'),
        QtdAlbuns=('AlbumID', 'nunique')
    )
    .reset_index()
    # Criando o percentual: dividindo a linha atual pela soma total da coluna
    .assign(
        PercentualCatalogo=lambda df: (df['QtdMusicas'] / df['QtdMusicas'].sum() * 100).round(2)
    )
    .sort_values(by='QtdMusicas', ascending=False)
)

# Exibindo o resultado no notebook
gravadoras_catalogo

,Gravadora,QtdMusicas,QtdAlbuns,PercentualCatalogo
7,Republic Records,26,6,20.97
5,Interscope Records,25,6,20.16
1,Atlantic Records,16,4,12.90
2,Columbia Records,12,3,9.68
9,Top Dawg Entertainment,12,3,9.68
11,Young Money Entertainment,8,2,6.45
10,Warner Music Group,5,1,4.03
0,Aftermath Entertainment,4,1,3.23
3,Def Jam Recordings,4,1,3.23
4,Epic Records,4,1,3.23


## 8. Quantos novos usuários foram cadastrados por mês?

**Pergunta de negócio:** Qual é o ritmo de aquisição de usuários?

```sql
SELECT YEAR(u.DateCreated) AS Ano,
    MONTH(u.DateCreated) AS Mes,
    COUNT(u.UserID) AS NovosUsuarios
FROM dbo.Users u
GROUP BY YEAR(u.DateCreated),
    MONTH(u.DateCreated)
ORDER BY Ano, Mes;
```

**Por que importa:** curva de crescimento da base - métrica fundamental para avaliar saúde da plataforma.


In [7]:
aquisicao_usuarios = (
    df_users
    .assign(
        Ano=lambda df: pd.to_datetime(df['DateCreated']).dt.year,
        Mes=lambda df: pd.to_datetime(df['DateCreated']).dt.month
    )
    .groupby(['Ano', 'Mes'])
    .agg(NovosUsuarios=('UserID', 'count'))
    .reset_index()
    .sort_values(by=['Ano', 'Mes'])
    )

aquisicao_usuarios

,Ano,Mes,NovosUsuarios
0,2018,1,4
1,2018,2,8
2,2018,3,8
3,2018,4,6
4,2018,5,11
...,...,...,...
79,2024,8,6
80,2024,9,9
81,2024,10,4
82,2024,11,11


## 9. Qual o horário de pico de reproduções?

**Pergunta de negócio:** Em que horas do dia os usuários mais consomem música?

```sql
SELECT DATEPART(HOUR, sp.StartTime) AS Hora,
    COUNT(sp.SongPlayID) AS TotalReproducoes
FROM dbo.SongPlays sp
GROUP BY DATEPART(HOUR, sp.StartTime)
ORDER BY TotalReproducoes DESC;
```

**Por que importa:** define janelas ideais para lançamentos, notificações e campanhas de marketing.


In [4]:
horario_pico = (
    df_songplays
    .assign(Hora=lambda df: pd.to_datetime(df['StartTime']).dt.hour)
    .groupby('Hora')
    .agg(TotalReproducoes=('SongPlayID', 'count'))
    .reset_index()
    .sort_values(by='TotalReproducoes', ascending=False)
)

horario_pico

,Hora,TotalReproducoes
11,11,366
15,15,355
19,19,355
1,1,354
22,22,354
17,17,350
6,6,346
7,7,342
9,9,341
16,16,341


## 10. Qual a duração média das sessões de escuta por país?

**Pergunta de negócio:** Usuários de quais países passam mais tempo na plataforma?

```sql
SELECT l.Country AS Pais,
    COUNT(sp.SongPlayID) AS TotalReproducoes,
    AVG(DATEDIFF(SECOND, sp.StartTime, sp.EndTime)) / 60 AS DuracaoMediaMinutos
FROM dbo.SongPlays sp
    JOIN dbo.Location l ON l.LocationID = sp.LocationID
GROUP BY l.Country
ORDER BY DuracaoMediaMinutos DESC;
```

**Por que importa:** tempo de sessão é indicador de engajamento real, não só de cliques.

In [ ]:
duracao_sessao_pais = (
    pd.merge(df_songplays, df_location, on='LocationID', how='inner')
    .assign(
        Inicio=lambda df: pd.to_datetime(df['StartTime']),
        Fim=lambda df: pd.to_datetime(df['EndTime']),
        DuracaoMinutos=lambda df: (df['Fim'] - df['Inicio']).dt.total_seconds() / 60
    )
    .groupby('Country')
    .agg(
        TotalReproducoes=('SongPlayID', 'count'),
        DuracaoMediaMinutos=('DuracaoMinutos', 'mean')
    )
    .reset_index()
    .rename(columns= {'Country' : 'Pais'})
    .assign(DuracaoMediaMinutos=lambda df: df['DuracaoMediaMinutos'].round(2))
    .sort_values(by='DuracaoMediaMinutos', ascending=False)
)

duracao_sessao_pais

,Pais,TotalReproducoes,DuracaoMediaMinutos
19,United States,383,3.52
7,Germany,412,3.50
15,South Africa,427,3.50
13,Peru,378,3.49
12,Nigeria,400,3.46
8,India,421,3.46
0,Argentina,397,3.46
9,Japan,419,3.44
10,Mexico,443,3.44
3,Canada,362,3.42
